In [64]:
import requests
import pandas as pd
import unittest

BASE_URL = "http://127.0.0.1:8000"

# ========== Fetch data ==========
def fetch_users():
    return requests.get(f"{BASE_URL}/users").json()
    
def fetch_tracks():
    return requests.get(f"{BASE_URL}/tracks").json()
    
def fetch_listening_history():
    return requests.get(f"{BASE_URL}/listen_history").json()  


In [65]:
# ========== Data exploration ==========
users_data = fetch_users()
user_items = users_data["items"]
df_users   = pd.DataFrame(user_items)
df_users.head(5)

,id,first_name,last_name,email,gender,favorite_genres,created_at,updated_at
0,79131,Shelby,Chapman,smithjessica@example.net,Gender nonconforming,R&B,2024-07-15T05:27:16,2024-09-02T01:15:48
1,94770,Pamela,Fisher,lindseylopez@example.org,Agender,Metal,2024-10-11T15:12:33,2025-05-17T00:16:12
2,7281,Jillian,Garcia,wisetimothy@example.org,Bigender,Country,2024-04-27T21:47:20,2025-02-02T04:55:38
3,41322,Peter,Ford,lthomas@example.net,Female,Metal,2024-05-15T12:23:45,2024-12-13T01:10:39
4,62096,Jessica,Wilson,alison63@example.org,Male,Reggae,2023-11-21T02:09:02,2025-04-12T12:02:42


In [66]:
tracks = fetch_tracks()
tracks  = tracks["items"]
df_tracks     = pd.DataFrame(tracks)
df_tracks.head(5)

,id,name,artist,songwriters,duration,genres,album,created_at,updated_at
0,71400,wrong,Lucas Jackson,Brian Briggs,03:36,open,technology,2023-12-30T00:31:27,2024-12-04T14:25:35
1,29953,must,Amanda Baker,Daniel Nielsen,11:47,another,paper,2023-08-14T05:33:15,2024-08-06T05:53:13
2,98175,force,Justin Pitts,Jennifer Moore,24:39,this,kid,2024-10-18T14:57:09,2024-08-07T20:09:19
3,16784,he,Jennifer Davis,Jasmine Long,09:23,put,wonder,2025-02-26T16:44:54,2025-02-27T20:37:12
4,83948,general,David Johnson,Jessica Sherman,43:11,material,agreement,2025-03-30T14:17:40,2025-07-15T18:02:17


In [67]:
listening_data = fetch_listening_history()
history_items  = listening_data["items"]
df_history     = pd.DataFrame(history_items)
df_history.head(5)

,user_id,items,created_at,updated_at
0,79131,"[30106, 4418, 86588, 74446, 41068]",2025-07-09T17:16:19,2025-07-14T23:46:41
1,94770,"[54523, 43239, 86773, 45322, 57317]",2025-06-24T19:33:18,2025-07-01T14:59:08
2,7281,"[44837, 44669, 71988, 3316, 73023]",2024-05-23T13:20:56,2024-07-07T06:49:13
3,41322,"[82179, 48003, 46483, 35098, 18428]",2024-03-03T12:50:47,2025-06-17T19:24:17
4,62096,"[47124, 35217, 967, 31372, 91523]",2023-12-10T00:29:28,2024-03-26T05:20:09


In [68]:
# Flattening listening history data
History_flattened = []

for _, row in df_history.iterrows():
    user_id = row["user_id"]  
    for song_id in row["items"]:
        History_flattened.append({
            "user_id": user_id,
            "song_id": song_id,
        })
df_history_flat = pd.DataFrame(History_flattened)
print(df_history_flat.sample(5))


     user_id  song_id
190    75836    50721
459     8346    42445
383    66686    20642
360    60834    74762
119    90365    38520


In [69]:
# ========== Data quality ==========

# Checking for missing values 
missing_df = df_users.isnull().sum().reset_index()
missing_df.columns = ['Column', 'Missing Values']
missing_df.head(5)

,Column,Missing Values
0,id,0
1,first_name,0
2,last_name,0
3,email,0
4,gender,0


In [70]:
# Checking for duplicate user IDs
duplicate_ids = df_users["id"].duplicated().sum()
print(f"Number of duplicate user IDs: {duplicate_ids}")

# Check for users with no listening history
users_no_history = df_history[df_history["items"].apply(lambda x: len(x) == 0)]
no_history_count = len(users_no_history)

print(f"Number of users without listening history: {no_history_count}")
if no_history_count > 0:
    display(users_no_history.head())

# Checking for users whose creation date is later than their last update date 
anomaly_df = df_users[df_users["created_at"] > df_users["updated_at"]]
anomaly_count = len(anomaly_df)
print(f"Number of users with a creation date later than the update date: {anomaly_count}")
if anomaly_count > 0:
    display(anomaly_df.head())


Number of duplicate user IDs: 0
Number of users without listening history: 0
Number of users with a creation date later than the update date: 30


,id,first_name,last_name,email,gender,favorite_genres,created_at,updated_at
11,50390,Christopher,Davis,mortega@example.org,Non-binary,R&B,2025-03-21T18:42:29,2024-10-02T13:53:11
12,59746,Devin,Miranda,michael45@example.org,Genderfluid,Electronic,2025-07-09T21:24:44,2025-01-19T09:43:40
13,16085,Gregory,Parker,brandon36@example.net,Genderqueer,R&B,2025-07-16T23:35:47,2025-03-28T11:01:18
15,23933,Karen,Benitez,halldarren@example.org,Gender questioning,Metal,2025-05-01T02:11:51,2025-04-13T21:45:05
16,8061,Stephanie,Morrison,paigeholland@example.com,Agender,Hip Hop,2025-05-23T10:41:44,2024-09-19T15:29:39


In [71]:
# ========== Validate Songs ==========
try:
    songs_data = fetch_songs()
    items = songs_data.get("items", [])
    
    assert isinstance(items, list), "'items' should be a list"

    for i, song in enumerate(items):
        assert "id" in song, f"Missing 'id' in item {i}"
        assert isinstance(song["id"], int), f"'id' is not an int in item {i}"

        assert "name" in song, f"Missing 'name' in item {i}"
        assert isinstance(song["name"], str), f"'name' is not a string in item {i}"
        assert song["name"].strip() != "", f"'name' is empty in item {i}"
    
    print("Songs have valid 'id' and 'name'")
except AssertionError as e:
    print(f"Data quality test failed: {e}")
except Exception as e:
    print(f"Unexpected error during data quality test: {e}")


# ========== Validate Listening History Entries ==========
try:
    history_data = fetch_listening_history()
    items = history_data.get("items", [])
    
    assert isinstance(items, list), "Expected 'items' to be a list"

    for i, entry in enumerate(items):
        assert "user_id" in entry, f"Missing 'user_id' in item {i}"
        assert isinstance(entry["user_id"], int), f"'user_id' is not an int in item {i}"
        
        assert "items" in entry, f"Missing 'items' (song list) in item {i}"
        assert isinstance(entry["items"], list), f"'items' is not a list in item {i}"
        
        assert len(entry["items"]) > 0, f"'items' list is empty in item {i}"

        assert "created_at" in entry, f"Missing 'created_at' in item {i}"
        assert "updated_at" in entry, f"Missing 'updated_at' in item {i}"
    
    print("Listening history entries are valid")
except AssertionError as e:
    print(f"Data quality test failed: {e}")
except Exception as e:
    print(f"Unexpected error during data quality test: {e}")

Unexpected error during data quality test: name 'fetch_songs' is not defined
Listening history entries are valid


In [72]:
# ========== Unit Test ==========
# ========== Test 1: API returns a response (status 200) ==========
try:
    response = requests.get(f"{BASE_URL}/users")
    assert response.status_code == 200, "API /users not reachable"
    print("Test 1 passed: API /users is reachable")
except Exception as e:
    print("Test 1 failed:", e)

# ========== Test 2: Response is a JSON with expected structure ==========
try:
    users_data = fetch_users()
    assert isinstance(users_data, dict), "/users response is not a dictionary"
    assert "items" in users_data, "Key 'items' not found in /users response"
    assert isinstance(users_data["items"], list), "'items' is not a list"
    print("Test 2 passed: /users returns a valid JSON structure")
except Exception as e:
    print("Test 2 failed:", e)

# ========== Test 3: Required fields exist in first user ==========
try:
    user = users_data["items"][0]
    assert "id" in user, "'id' missing in user"
    assert "email" in user, "'email' missing in user"
    assert isinstance(user["id"], int), "'id' is not an integer"
    assert isinstance(user["email"], str), "'email' is not a string"
    print("Test 3 passed: User contains required fields with correct types")
except Exception as e:
    print("Test 3 failed:", e)

# ========== Test 4: Listening history is valid ==========
try:
    history_data = fetch_listening_history()
    assert "items" in history_data, "Key 'items' missing in /listen_history"
    assert isinstance(history_data["items"], list), "Listening history items not a list"
    print("Test 4 passed: Listening history structure is valid")
except Exception as e:
    print("Test 4 failed:", e)

# ========== Test 5: Songs endpoint is working and structured ==========
try:
    songs_data = fetch_songs()
    assert isinstance(songs_data, dict), "Songs response not a dictionary"
    assert "items" in songs_data, "Key 'items' not in /songs"
    assert isinstance(songs_data["items"], list), "Songs 'items' is not a list"
    print("Test 5 passed: Songs endpoint returns valid structure")
except Exception as e:
    print("Test 5 failed:", e)


Test 1 passed: API /users is reachable
Test 2 passed: /users returns a valid JSON structure
Test 3 passed: User contains required fields with correct types
Test 4 passed: Listening history structure is valid
Test 5 failed: name 'fetch_songs' is not defined
